# 3D-ΔPDF: analysis-only prototype

**This is a synthetic volume, not DISCUS output.** A quantitative 3D-ΔPDF requires a Bragg-subtracted diffuse intensity volume on a corrected, centered Cartesian Q grid; Bragg subtraction, masking, normalization, and voxel interpolation are outside this starter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from discuspdf import delta_pdf_from_diffuse, plot_delta_pdf_slice, FFTValidationError

## 1. Construct a synthetic centrosymmetric diffuse intensity

Coordinates below are in physical Q units Å⁻¹.

In [ ]:
n = 64
dq = 0.15  # Å^-1
q = (np.arange(n) - n//2) * dq
qx, qy, qz = np.meshgrid(q, q, q, indexing="ij")
# An example smooth, nonnegative centrosymmetric feature
i_diff = np.exp(-((qx-1.1)**2+qy**2+qz**2)/(2*0.35**2))
i_diff += np.exp(-((qx+1.1)**2+qy**2+qz**2)/(2*0.35**2))

## 2. Transform to a real-space 3D-ΔPDF

In [ ]:
fft = delta_pdf_from_diffuse(i_diff, (q, q, q))
print(fft.diagnostics)
delta, r_axes = fft.real_volume, fft.r_axes
ax = plot_delta_pdf_slice(delta, r_axes, axis=2)
plt.show()

## 3. Replace synthetic data with your own

For an HDF5 file, identify the actual dataset paths first. **Do not assume** a universal DISCUS HDF5 layout.

```python
from discuspdf import read_hdf5_volume, delta_pdf_from_diffuse, FFTValidationError
intensity, q_axes = read_hdf5_volume(
    "your_diffuse_volume.h5",
    intensity_dataset="/actual/intensity/path",
    q_datasets=("/actual/qx/path", "/actual/qy/path", "/actual/qz/path"),
)
try:
    fft = delta_pdf_from_diffuse(intensity, q_axes)
except FFTValidationError as exc:
    print(exc.result.diagnostics)  # the computed FFT is preserved on the exception
else:
    delta, r_axes = fft.real_volume, fft.r_axes
```

If the source uses h,k,l reciprocal-lattice coordinates, first map the data to an appropriately sampled Cartesian Q grid using the reciprocal-cell metric. This notebook never runs a real DISCUS 3D computation; see `docs/3D_INTEGRATION.md` for the runner workflow.